# App Function Tests

This notebook lets you test specific functions from `app.py` without changing the app itself.

Notes:
- `get_route()` uses OpenRouteService and needs `ORS_API_KEY` set.
- The TfL helpers do not need your ORS key.
- Coordinates in your app are stored as `[lng, lat]`.

In [43]:
100 // 15

6

In [2]:
import os
from pprint import pprint

import app as commute_app

print("ORS_API_KEY present:", bool(os.getenv("ORS_API_KEY")))
print("Imported functions:")
print([
    name for name in dir(commute_app)
    if name.startswith("get_")
])

ORS_API_KEY present: False
Imported functions:
['get_isochrone', 'get_point_at_distance', 'get_route', 'get_route_data', 'get_tfl_duration', 'get_tfl_isochrone', 'get_tfl_route']


In [ ]:
# Sample coordinates in [lng, lat] format
WESTMINSTER = [-0.1246, 51.5007]
BUCKINGHAM_PALACE = [-0.1419, 51.5014]
KINGS_CROSS = [-0.1238, 51.5308]
CANARY_WHARF = [-0.0194, 51.5054]
HOME = [-0.142302, 51.405202]


print("WESTMINSTER:", WESTMINSTER)
print("KINGS_CROSS:", KINGS_CROSS)

WESTMINSTER: [-0.1246, 51.5007]
KINGS_CROSS: [-0.1238, 51.5308]


## Pure helper test

In [26]:
# Test the geometry helper without hitting any API
point = commute_app.get_point_at_distance(
    origin_lng=HOME[0],
    origin_lat=HOME[1],
    bearing_deg=270,
    distance_km=50,
)
print("Point 50km west of home:")
print(point)

Point 50km west of home:
(-0.8631100881319299, 51.402991323687466)


## TfL direct function tests

In [27]:
# Test a public transport route between two points
tfl_route = commute_app.get_tfl_route(HOME, point)
pprint(tfl_route)

TFL ERROR: 'lon'
Raw response: {'$type': 'Tfl.Api.Presentation.Entities.JourneyPlanner.ItineraryResult, Tfl.Api.Presentation.Entities', 'journeys': [{'$type': 'Tfl.Api.Presentation.Entities.JourneyPlanner.Journey, Tfl.Api.Presentation.Entities', 'startDateTime': '2026-04-22T05:13:00', 'duration': 145, 'arrivalDateTime': '2026-04-22T07:38:00', 'alternativeRoute': False, 'legs': [{'$type': 'Tfl.Api.Presentation.Entities.JourneyPlanner.Leg, Tfl.Api.Presentation.Entities', 'duration': 5, 'instruction': {'$type': 'Tfl.Api.Presentation.Entities.Instruction, Tfl.Api.Presentation.Entities', 'summary': 'Walk to Stanford Way', 'detailed': 'Walk to Stanford Way', 'steps': [{'$type': 'Tfl.Api.Presentation.Entities.InstructionStep, Tfl.Api.Presentation.Entities', 'description': ' for 28 metres', 'turnDirection': 'STRAIGHT', 'streetName': '', 'distance': 28, 'cumulativeDistance': 28, 'skyDirection': 107, 'skyDirectionDescription': 'East', 'cumulativeTravelTime': 23, 'latitude': 51.405150448633, 'lon

In [34]:
# Test just the TfL journey duration between two coordinates
point = commute_app.get_point_at_distance(
    origin_lng=HOME[0],
    origin_lat=HOME[1],
    bearing_deg=270,
    distance_km=5,
)
print("Point 50km west of home:")
print(point)

tfl_duration = commute_app.get_tfl_duration(
    HOME[0], HOME[1],
    point[0], point[1],
)
print("TfL duration (minutes):", tfl_duration)

Point 50km west of home:
(-0.21438510857276383, 51.40517989259555)
TfL duration (minutes): 49


In [31]:
(tfl_polygon)

[[-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.11580790402581907, 51.40737476334469],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202],
 [-0.142302, 51.405202]]

In [29]:
# Test the TfL isochrone helper
# This can take a while because it makes many TfL calls.
minutes = 20
tfl_polygon = commute_app.get_tfl_isochrone(HOME, minutes)
print("Returned points:", len(tfl_polygon) if tfl_polygon else 0)
print("First 3 points:")
pprint(tfl_polygon[:3] if tfl_polygon else None)

Returned points: 25
First 3 points:
[[-0.142302, 51.405202], [-0.142302, 51.405202], [-0.142302, 51.405202]]


## OpenRouteService direct function test

In [10]:
# Requires ORS_API_KEY in your environment
ors_route = commute_app.get_route(WESTMINSTER, BUCKINGHAM_PALACE, mode="driving-car")
pprint(ors_route)

ORS ERROR: 'routes'
Raw response: {'error': 'Authorization field missing'}
None


## Flask endpoint tests

In [ ]:
client = commute_app.app.test_client()

response = client.post(
    "/route",
    json={
        "start": [51.404938, -0.14198],
        "end": KINGS_CROSS,
        "mode": "public-transport",
    },
)

print("Status:", response.status_code)
pprint(response.get_json())

TFL ERROR: 'lon'
Raw response: {'$type': 'Tfl.Api.Presentation.Entities.JourneyPlanner.ItineraryResult, Tfl.Api.Presentation.Entities', 'journeys': [{'$type': 'Tfl.Api.Presentation.Entities.JourneyPlanner.Journey, Tfl.Api.Presentation.Entities', 'startDateTime': '2026-04-21T16:25:00', 'duration': 27, 'arrivalDateTime': '2026-04-21T16:52:00', 'alternativeRoute': False, 'legs': [{'$type': 'Tfl.Api.Presentation.Entities.JourneyPlanner.Leg, Tfl.Api.Presentation.Entities', 'duration': 5, 'instruction': {'$type': 'Tfl.Api.Presentation.Entities.Instruction, Tfl.Api.Presentation.Entities', 'summary': 'Walk to Westminster Station', 'detailed': 'Walk to Westminster Station', 'steps': [{'$type': 'Tfl.Api.Presentation.Entities.InstructionStep, Tfl.Api.Presentation.Entities', 'description': 'Bridge Street for 16 metres', 'turnDirection': 'STRAIGHT', 'streetName': 'Bridge Street', 'distance': 16, 'cumulativeDistance': 16, 'skyDirection': 269, 'skyDirectionDescription': 'West', 'cumulativeTravelTime'

In [12]:
client = commute_app.app.test_client()

response = client.post(
    "/isochrone",
    json={
        "coord": WESTMINSTER,
        "mode": "public-transport",
        "minutes": 20,
    },
)

print("Status:", response.status_code)
data = response.get_json()
print("Polygon points:", len(data["polygon"]) if data and "polygon" in data else 0)
pprint(data)

Status: 200
Polygon points: 25
{'polygon': [[-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.11000235297186182, 51.50767081180482],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007],
             [-0.1246, 51.5007]]}


## Quick edit cell

Use this cell to swap in any coordinates you want to test.

In [19]:
# Replace these with your own clicked/map coordinates in [lng, lat] format
START =[51.404938, -0.14198]
END = [51.50554656982422,-0.21776099503040314]
MINUTES = 60

print("START:", START)
print("END:", END)
print("MINUTES:", MINUTES)

START: [51.404938, -0.14198]
END: [51.50554656982422, -0.21776099503040314]
MINUTES: 60


In [34]:
import csv

def load_stops_from_csv(filename="naptan_2026_london.csv"):
    stops = []

    try:
        with open(filename, newline="", encoding="cp1252", errors="ignore") as f:
            reader = csv.DictReader(f)

            for row in reader:
                try:
                    lng = float(row["Longitude"])
                    lat = float(row["Latitude"])

                    # skip empty or invalid coords
                    if lng == 0 or lat == 0:
                        continue

                    stops.append([lng, lat])

                except (ValueError, KeyError, TypeError):
                    continue

    except Exception as e:
        print("CSV LOAD ERROR:", e)

    return stops


# IMPORTANT: actually call it
stops = load_stops_from_csv()

print("Loaded stops:", len(stops))
print("Sample:", stops[:5])

Loaded stops: 19601
Sample: [[-0.07333, 51.51283], [-0.0711, 51.51439], [-0.07528, 51.51405], [-0.07493, 51.51408], [-0.07474, 51.51429]]


In [19]:
def get_nearby_stops(origin_lng, origin_lat, radius_metres=10000):
    nearby = []

    for stop_lng, stop_lat in STOPS:
        # simple distance calc (Haversine)
        dx = math.radians(stop_lat - origin_lat)
        dy = math.radians(stop_lng - origin_lng)

        a = (math.sin(dx/2)**2 +
             math.cos(math.radians(origin_lat)) *
             math.cos(math.radians(stop_lat)) *
             math.sin(dy/2)**2)

        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

        distance_km = 6371 * c
        distance_m = distance_km * 1000

        if distance_m <= radius_metres:
            nearby.append([stop_lng, stop_lat])

    return nearby

SyntaxError: invalid syntax (3816831728.py, line 1)

In [33]:
import pandas as pd

df = pd.read_csv("naptan_2026_london.csv", encoding="cp1252")

print (df.columns)

Index(['ATCOCode', 'NaptanCode', 'CommonName', 'Street', 'Indicator',
       'Bearing', 'NptgLocalityCode', 'LocalityName', 'ParentLocalityName',
       'Town', 'GridType', 'Easting', 'Northing', 'Longitude', 'Latitude',
       'StopType', 'BusStopType', 'TimingStatus', 'AdministrativeAreaCode',
       'CreationDateTime', 'ModificationDateTime', 'RevisionNumber',
       'Modification', 'Status'],
      dtype='str')


In [46]:
#check if code reading the CSV correctly

def load_stops_from_csv(filename="naptan_2026_london.csv"):
    stops = []
    try:
        with open(filename, newline="", encoding="cp1252", errors="ignore") as f:
            reader = csv.DictReader(f)
            for row in reader:
                try:
                    lng = float(row["Longitude"])
                    lat = float(row["Latitude"])
                    mode = row.get("Mode", "").strip().lower()  # ← read mode column
                    if lng != 0 and lat != 0:
                        stops.append([lng, lat, mode])              # ← store mode
                except (ValueError, KeyError, TypeError):
                    continue
    except Exception as e:
        print("CSV LOAD ERROR:", e)
    return stops

STOPS = load_stops_from_csv()
print(f"Loaded {len(STOPS)} stops from CSV")
# temporary - print unique mode values to confirm column name
print("Mode values in CSV:", set(s[2] for s in STOPS))

Loaded 19601 stops from CSV
Mode values in CSV: {'national-rail', 'bus', 'airplane', 'tube', 'ferry'}
